# 03 - Replay-based strategies: Replay, GEM, A-GEM, GDumb

These strategies keep a small memory buffer of past examples:

- **Replay**: interleaves buffered past examples with new-experience
  minibatches during training.
- **GEM** (Gradient Episodic Memory): uses the buffer to *constrain*
  gradient updates so loss on past examples never increases.
- **A-GEM**: a cheaper, averaged relaxation of GEM's constraint.
- **GDumb**: greedily maintains a class-balanced buffer and simply
  retrains a fresh model on the buffer alone at eval time -- a
  deceptively strong, very simple baseline.
- **ER-ACE** (Experience Replay with Asymmetric Cross-Entropy): not in
  stock Avalanche. Ported from
  [`AlbinSou/ocl_survey`](https://github.com/AlbinSou/ocl_survey)
  (Caccia et al., ICLR 2022) -- see `src/cl_bench/er_ace.py`. Uses an
  asymmetric loss so the *current* minibatch's loss only pushes down
  logits for classes actually present in that minibatch, avoiding an
  "abrupt representation change" that plain Replay is prone to.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import warnings; warnings.filterwarnings("ignore")

import torch
import pandas as pd
from avalanche.models import SimpleMLP
from avalanche.training import Replay, GEM, AGEM, GDumb
from avalanche.training.plugins import EvaluationPlugin
from avalanche.evaluation.metrics import accuracy_metrics, forgetting_metrics, loss_metrics

from bench_utils import make_synthetic_benchmark
from run_utils import run_strategy
from cl_bench.er_ace import ER_ACE

BENCHMARK_CONFIG = dict(
    n_classes=10, n_experiences=5, feature_dim=64,
    n_per_class=250, class_sep=1.6, noise=1.0, seed=0,
)
benchmark = make_synthetic_benchmark(**BENCHMARK_CONFIG)

def new_model():
    return SimpleMLP(num_classes=benchmark.n_classes, input_size=benchmark.feature_dim,
                      hidden_size=64, hidden_layers=1, drop_rate=0.0)

def new_evaluator():
    return EvaluationPlugin(
        accuracy_metrics(experience=True, stream=True),
        forgetting_metrics(experience=True, stream=True),
        loss_metrics(stream=True),
        loggers=[],
    )

all_rows = []


In [2]:
# --- Replay ---
model = new_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
strategy = Replay(model=model, optimizer=opt, criterion=torch.nn.CrossEntropyLoss(),
                   mem_size=200, train_mb_size=32, train_epochs=3, eval_mb_size=128,
                   evaluator=new_evaluator(), device="cpu")
rows, final = run_strategy(strategy, benchmark, "Replay", "replay")
all_rows += rows
print(final)


/usr/local/lib/python3.12/dist-packages/avalanche/training/plugins/replay.py:123: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)
/usr/local/lib/python3.12/dist-packages/avalanche/training/plugins/replay.py:123: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)
/usr/local/lib/python3.12/dist-packages/avalanche/training/plugins/replay.py:123: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)


{'strategy': 'Replay', 'category': 'replay', 'after_experience': 4, 'stream_acc': 0.996, 'stream_forgetting': 0.004807692307692318, 'train_seconds': 0.3852355480194092}


/usr/local/lib/python3.12/dist-packages/avalanche/training/plugins/replay.py:123: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)
/usr/local/lib/python3.12/dist-packages/avalanche/training/plugins/replay.py:123: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)


In [3]:
# --- GEM ---
model = new_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
strategy = GEM(model=model, optimizer=opt, criterion=torch.nn.CrossEntropyLoss(),
               patterns_per_exp=64, train_mb_size=32, train_epochs=3, eval_mb_size=128,
               evaluator=new_evaluator(), device="cpu")
rows, final = run_strategy(strategy, benchmark, "GEM", "replay")
all_rows += rows
print(final)


{'strategy': 'GEM', 'category': 'replay', 'after_experience': 4, 'stream_acc': 0.994, 'stream_forgetting': 0.006880733944954115, 'train_seconds': 0.530189037322998}


In [4]:
# --- A-GEM ---
model = new_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
strategy = AGEM(model=model, optimizer=opt, criterion=torch.nn.CrossEntropyLoss(),
                patterns_per_exp=64, sample_size=64, train_mb_size=32, train_epochs=3,
                eval_mb_size=128, evaluator=new_evaluator(), device="cpu")
rows, final = run_strategy(strategy, benchmark, "A-GEM", "replay")
all_rows += rows
print(final)


{'strategy': 'A-GEM', 'category': 'replay', 'after_experience': 4, 'stream_acc': 0.35, 'stream_forgetting': 0.7983837970540099, 'train_seconds': 0.5252115726470947}


In [5]:
# --- GDumb ---
model = new_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
strategy = GDumb(model=model, optimizer=opt, criterion=torch.nn.CrossEntropyLoss(),
                  mem_size=200, train_mb_size=32, train_epochs=3, eval_mb_size=128,
                  evaluator=new_evaluator(), device="cpu")
rows, final = run_strategy(strategy, benchmark, "GDumb", "replay")
all_rows += rows
print(final)


{'strategy': 'GDumb', 'category': 'replay', 'after_experience': 4, 'stream_acc': 1.0, 'stream_forgetting': 0.0, 'train_seconds': 0.1994791030883789}


/usr/local/lib/python3.12/dist-packages/avalanche/training/plugins/gdumb.py:47: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)
/usr/local/lib/python3.12/dist-packages/avalanche/training/plugins/gdumb.py:47: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)
/usr/local/lib/python3.12/dist-packages/avalanche/training/plugins/gdumb.py:47: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)
/usr/local/lib/python3.12/dist-packages/avalanche/training/plugins/gdumb.py:47: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)
/usr/local/lib/pytho

In [6]:
# --- ER-ACE ---
model = new_model()
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
strategy = ER_ACE(model=model, optimizer=opt, batch_size_mem=32, mem_size=200, alpha=0.5,
                   train_mb_size=32, train_epochs=3, eval_mb_size=128,
                   evaluator=new_evaluator(), device="cpu")
rows, final = run_strategy(strategy, benchmark, "ER-ACE", "replay")
all_rows += rows
print(final)


/home/claude/project/src/cl_bench/er_ace.py:163: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(self, **kwargs)
/home/claude/project/src/cl_bench/er_ace.py:163: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(self, **kwargs)
/home/claude/project/src/cl_bench/er_ace.py:163: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(self, **kwargs)


{'strategy': 'ER-ACE', 'category': 'replay', 'after_experience': 4, 'stream_acc': 1.0, 'stream_forgetting': 0.0, 'train_seconds': 0.44919657707214355}


/home/claude/project/src/cl_bench/er_ace.py:163: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(self, **kwargs)
/home/claude/project/src/cl_bench/er_ace.py:163: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(self, **kwargs)


In [7]:
df = pd.DataFrame(all_rows)
df.to_csv("../results/03_replay.csv", index=False)
df


,strategy,category,after_experience,stream_acc,stream_forgetting
0,Replay,replay,0,0.218,0.000000
1,Replay,replay,1,0.406,0.000000
2,Replay,replay,2,0.614,0.000000
3,Replay,replay,3,0.802,0.000000
4,Replay,replay,4,0.996,0.004808
5,GEM,replay,0,0.218,0.000000
6,GEM,replay,1,0.406,0.000000
7,GEM,replay,2,0.614,0.000000
8,GEM,replay,3,0.802,0.000000
9,GEM,replay,4,0.994,0.006881
